# 05 — Summarize

Assembles the final comparison table with ΔRMSE vs each model's own random-split RMSE. Writes `results/comparison_stage2.md`.

In [ ]:
from pathlib import Path
import pandas as pd

RES = Path.cwd().parent / 'results'
df = pd.read_csv(RES / 'metrics.csv')
df

## Reference RMSE per model (random split, seed 42)

In [ ]:
ref = df.query("split == 'random'").set_index('model')['test_rmse'].to_dict()
ref

## Full ΔRMSE table

In [ ]:
SPLIT_ORDER = ['random', 'textrap', 'coldsol', 'coldpair', 'coldsolv']
SPLIT_LABEL = {'random': 'random', 'textrap': 'T-extrap',
               'coldsol': 'cold-solute', 'coldpair': 'cold-pair', 'coldsolv': 'cold-solvent'}
MODEL_ORDER = ['A', 'D', 'B']
MODEL_LABEL = {'A': 'A direct', 'D': 'D 1/T control', 'B': "B Van't Hoff"}

rows = []
for m in MODEL_ORDER:
    for s in SPLIT_ORDER:
        r = df.query(f"model == '{m}' and split == '{s}'").iloc[0]
        rows.append({
            'model': MODEL_LABEL[m],
            'split': SPLIT_LABEL[s],
            'test_rmse': round(r['test_rmse'], 3),
            'test_r2':   round(r['test_r2'], 3),
            'delta_rmse': round(r['test_rmse'] - ref[m], 3) if s != 'random' else 0.0,
        })
summary = pd.DataFrame(rows)
summary

## Rankings

In [ ]:
for s in SPLIT_ORDER:
    sub = df.query(f"split == '{s}'").sort_values('test_rmse')
    order = ' -> '.join(f"{r['model']} ({r['test_rmse']:.3f})" for _, r in sub.iterrows())
    print(f'{SPLIT_LABEL[s]:14s}  absolute best-to-worst: {order}')

In [ ]:
for s in SPLIT_ORDER:
    if s == 'random':
        continue
    sub = df.query(f"split == '{s}'").copy()
    sub['delta'] = sub.apply(lambda r: r['test_rmse'] - ref[r['model']], axis=1)
    sub = sub.sort_values('delta')
    order = ' -> '.join(f"{r['model']} (Δ={r['delta']:+.3f})" for _, r in sub.iterrows())
    print(f'{SPLIT_LABEL[s]:14s}  smallest ΔRMSE -> largest: {order}')

## Write the Markdown comparison report

In [ ]:
lines = []
w = lines.append
w('# Stage-2 comparison — A vs D vs B on 5 splits (seed 42)')
w('')
w('| Model | Split | Test RMSE | Test R² | ΔRMSE vs random |')
w('|---|---|---:|---:|---:|')
for m in MODEL_ORDER:
    for s in SPLIT_ORDER:
        r = df.query(f"model == '{m}' and split == '{s}'").iloc[0]
        delta = r['test_rmse'] - ref[m]
        d_str = f'{delta:+.3f}' if s != 'random' else '  0.000'
        w(f'| {MODEL_LABEL[m]} | {SPLIT_LABEL[s]} | {r["test_rmse"]:.3f} | '
          f'{r["test_r2"]:.3f} | {d_str} |')
report = '\n'.join(lines)
(RES / 'comparison_stage2.md').write_text(report)
print('wrote', RES / 'comparison_stage2.md')
print()
print(report)